In [ ]:
# Databricks notebook source
# COMMAND ----------
# MAGIC %pip install sodapy

# COMMAND ----------
import re
import json
import traceback
from datetime import datetime, timezone
from pyspark.sql.functions import col, current_timestamp
from sodapy import Socrata

# Tag all Spark SQL queries for tracking
spark.sql("SET QUERY_TAGS['data_origin'] = 'open_data_portal'")
spark.sql("SET QUERY_TAGS['pipeline'] = 'open_data_portal'")


# COMMAND ----------
# Parameters
dbutils.widgets.text("domain", "opendata.maryland.gov", "Socrata Domain")
dbutils.widgets.text("catalog_name", "open_data_portal", "Catalog Name")
dbutils.widgets.text("schema_name", "bronze", "Target Schema Name")
dbutils.widgets.text("meta_schema", "meta", "Control Schema Name")
dbutils.widgets.text("payload", "{}", "JSON Task Payload")
dbutils.widgets.dropdown("verbose", "true", ["true", "false"], "Verbose Logging")

DOMAIN = dbutils.widgets.get("domain")
CATALOG_NAME = dbutils.widgets.get("catalog_name")
SCHEMA_NAME = dbutils.widgets.get("schema_name")
META_SCHEMA = dbutils.widgets.get("meta_schema")
VERBOSE = dbutils.widgets.get("verbose").lower() == "true"
MAX_TAGS = 40

def log(msg: str):
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {msg}")

def sanitize_table_name(name: str) -> str:
    clean = re.sub(r'[^a-zA-Z0-9_]', '_', (name or '').lower())
    clean = re.sub(r'_+', '_', clean).strip('_')
    return clean[:200]

def sql_str(val) -> str:
    if val is None:
        return ""
    clean = str(val).replace("\\", "\\\\").replace("'", "''")
    return clean.replace("\r\n", "\\n").replace("\n", "\\n").replace("\r", "\\n")

def sanitize_tag_key(val: str) -> str:
    if not val:
        return ""
    clean = re.sub(r'[^a-zA-Z0-9_]', '_', str(val).strip().lower())
    clean = re.sub(r'_+', '_', clean).strip('_')
    return clean[:255]

def to_iso(val):
    if not val:
        return ""
    if isinstance(val, datetime):
        return val.isoformat()
    try:
        return datetime.fromtimestamp(int(val), tz=timezone.utc).isoformat()
    except Exception:
        return str(val)

def to_dt(ts):
    if not ts:
        return None
    try:
        return datetime.fromtimestamp(int(ts), tz=timezone.utc)
    except Exception:
        return None

def parse_catalog_timestamp(ts_val):
    if not ts_val:
        return None
    if isinstance(ts_val, (int, float)):
        try:
            return datetime.fromtimestamp(int(ts_val), tz=timezone.utc)
        except Exception:
            return None
    if isinstance(ts_val, str):
        cleaned_str = ts_val.strip().replace("Z", "+00:00")
        try:
            return datetime.fromisoformat(cleaned_str)
        except Exception:
            pass
        try:
            return datetime.fromtimestamp(int(cleaned_str), tz=timezone.utc)
        except Exception:
            return None
    return None

def append_log_event(
    dataset_id: str, 
    title: str, 
    table_name: str, 
    category: str, 
    portal_meta_update: str, 
    status: str, 
    records: int = 0, 
    error: str = ""
):
    """
    Appends an immutable audit row to the queue table.
    Wrapped in a try/except so logging crashes don't mask underlying exceptions.
    """
    try:
        from pyspark.sql.types import StructType, StructField, StringType, LongType
        
        schema = StructType([
            StructField("dataset_id", StringType(), False),
            StructField("task_type", StringType(), True),
            StructField("title", StringType(), True),
            StructField("table_name", StringType(), True),
            StructField("category", StringType(), True),
            StructField("portal_data_updated_at", StringType(), True),
            StructField("portal_metadata_updated_at", StringType(), True),
            StructField("status", StringType(), True),
            StructField("records_ingested", LongType(), True),
            StructField("error_message", StringType(), True)
        ])

        resolved_meta_update = portal_meta_update or None

        log_row = [{
            "dataset_id": dataset_id or "UNKNOWN",
            "task_type": "METADATA",
            "title": title or dataset_id or "UNKNOWN",
            "table_name": table_name or "UNKNOWN",
            "category": category or "Uncategorized",
            "portal_data_updated_at": None,
            "portal_metadata_updated_at": resolved_meta_update,
            "status": status,
            "records_ingested": records,
            "error_message": str(error)[:1000] if error else None
        }]

        queue_table = f"`{CATALOG_NAME}`.`{META_SCHEMA}`.`_socrata_ingestion_queue`"

        (spark.createDataFrame(log_row, schema=schema)
            .withColumn("logged_at", current_timestamp())
            .write
            .format("delta")
            .mode("append")
            .option("mergeSchema", "true")
            .saveAsTable(queue_table))
            
    except Exception as log_err:
        # Fallback to driver log if Delta queue write encounters an engine/auth issue
        print(f"[FATAL LOGGING ERROR] Could not persist to queue: {log_err}")
        print(f"[ORIGINAL TASK ERROR PAYLOAD] ID={dataset_id}, Status={status}, Err={error}")

# COMMAND ----------
# Safe Initialization Defaults
DATASET_ID = "UNKNOWN"
PORTAL_METADATA_UPDATED_AT = ""
dataset_title = "UNKNOWN"
table_name = "UNKNOWN"
category = "Uncategorized"
client = None

try:
    # 1. Safe Widget & Payload Unpacking
    raw_payload = dbutils.widgets.get("payload")
    try:
        task_data = json.loads(raw_payload) if raw_payload else {}
    except Exception as json_err:
        raise ValueError(f"Malformed JSON in payload widget: {json_err}")

    DATASET_ID = task_data.get("id", "").strip()
    PORTAL_METADATA_UPDATED_AT = task_data.get("up", "").strip()

    if not DATASET_ID:
        raise ValueError("dataset_id inside payload must be provided for single-task execution.")

    # 2. Socrata Authentication & Client Init
    SECRET_SCOPE = "socrata_scope"
    try:
        APP_TOKEN = dbutils.secrets.get(scope=SECRET_SCOPE, key="app_token")
        log("Loaded Socrata app token from secret scope.")
    except Exception:
        APP_TOKEN = None
        log("Warning: Running unauthenticated (rate limits apply).")

    client = Socrata(DOMAIN, APP_TOKEN, timeout=60)

    # 3. Fetch Metadata
    dataset_metadata = client.get_metadata(DATASET_ID)
    dataset_title = dataset_metadata.get("name", DATASET_ID)
    table_name = sanitize_table_name(dataset_title)
    TARGET_TABLE_PATH = f"`{CATALOG_NAME}`.`{SCHEMA_NAME}`.`{table_name}`"
    log(f"Retrieved views metadata for {DATASET_ID}.")

    metadata_updated_at_dt = parse_catalog_timestamp(PORTAL_METADATA_UPDATED_AT)

    owner = dataset_metadata.get("owner") or {}
    custom_fields = (dataset_metadata.get("metadata") or {}).get("custom_fields") or {}
    approvals = dataset_metadata.get("approvals") or []

    # 4. Log Metadata to Registry Table
    registry_record = [(
        DATASET_ID,
        DOMAIN,
        CATALOG_NAME,
        SCHEMA_NAME,
        table_name,
        dataset_metadata.get("name"),
        dataset_metadata.get("category"),
        dataset_metadata.get("license", {}).get("name") or dataset_metadata.get("licenseId"),
        owner.get("displayName"),
        owner.get("id"),
        dataset_metadata.get("provenance"),
        dataset_metadata.get("publicationStage"),
        to_dt(dataset_metadata.get("createdAt")),
        None,
        metadata_updated_at_dt,
        json.dumps(approvals),
        json.dumps(custom_fields),
        json.dumps(dataset_metadata),
        datetime.now(timezone.utc)
    )]

    reg_schema = """
        dataset_id STRING, domain STRING, catalog_name STRING, schema_name STRING, table_name STRING,
        dataset_title STRING, category STRING, license STRING, owner_name STRING, owner_id STRING,
        provenance STRING, publication_stage STRING,
        socrata_created_at TIMESTAMP, socrata_data_updated_at TIMESTAMP, socrata_metadata_updated_at TIMESTAMP,
        approvals STRING, custom_fields STRING, raw_metadata STRING, ingested_at TIMESTAMP
    """
    spark.createDataFrame(registry_record, schema=reg_schema) \
        .write.mode("append") \
        .saveAsTable(f"`{CATALOG_NAME}`.`{META_SCHEMA}`._socrata_metadata_registry")

    log(f"Logged dataset metadata to `{META_SCHEMA}`._socrata_metadata_registry.")

    # 5. Table Comment & TBLPROPERTIES
    description = dataset_metadata.get("description") or ""
    time_period_custom = custom_fields.get("Time Period") or {}
    agency_custom = custom_fields.get("Agency") or {}
    jurisdiction_custom = custom_fields.get("Jurisdiction") or {}
    data_provided_by = dataset_metadata.get("attribution") or ""
    category = dataset_metadata.get("category") or "Uncategorized"
    license_name = dataset_metadata.get("license", {}).get("name") or dataset_metadata.get("licenseId") or ""

    spark.sql(f"COMMENT ON TABLE {TARGET_TABLE_PATH} IS '{sql_str(description)}'")

    tbl_properties = {
        "open_data.title": dataset_metadata.get("name"),
        "open_data.url": f"https://{DOMAIN}/d/{DATASET_ID}",
        "open_data.attribution_url": dataset_metadata.get("attributionLink"),
        "open_data.category": category,
        "open_data.license": license_name,
        "open_data.agency": agency_custom.get("State Agency Performing Data Updates", data_provided_by),
        "open_data.jurisdiction": jurisdiction_custom.get("Jurisdiction", "State of Maryland"),
        "open_data.update_frequency": time_period_custom.get("Update Frequency"),
        "open_data.time_period_of_content": time_period_custom.get("Time Period of Content"),
        "open_data.owner": owner.get("displayName"),
        "open_data.owner_id": owner.get("id"),
        "open_data.provenance": dataset_metadata.get("provenance"),
        "open_data.publication_stage": dataset_metadata.get("publicationStage"),
        "open_data.created_at": to_iso(dataset_metadata.get("createdAt")),
        "open_data.metadata_updated_at": to_iso(metadata_updated_at_dt),
        "socrata.domain": DOMAIN,
        "socrata.dataset_id": DATASET_ID,
        "socrata.table_id": str(dataset_metadata.get("tableId", "")),
        "socrata.publication_group": str(dataset_metadata.get("publicationGroup", "")),
        "source.system": "Socrata Open Data Portal"
    }

    prop_clauses = [f"'{sql_str(k)}' = '{sql_str(v)}'" for k, v in tbl_properties.items() if sql_str(v)]
    if prop_clauses:
        props_sql = ",\n    ".join(prop_clauses)
        spark.sql(f"ALTER TABLE {TARGET_TABLE_PATH} SET TBLPROPERTIES (\n    {props_sql}\n)")
        log("Applied filtered TBLPROPERTIES.")

    # 6. Apply Unity Catalog Table Tags
    raw_tags = dataset_metadata.get("tags") or []
    tag_candidates = [sanitize_tag_key(t) for t in list(raw_tags) + [category]]
    unique_simple_tags = list(dict.fromkeys([t for t in tag_candidates if t]))

    tag_assignments = []
    key_value_tags = {
        "lifecycle": "current",
        "source_system": "open_data_portal",
        "domain": sanitize_tag_key(DOMAIN),
        "socrata_id": sanitize_tag_key(DATASET_ID),
        "category": sanitize_tag_key(category),
        "provenance": sanitize_tag_key(dataset_metadata.get("provenance", "")),
    }

    if agency_custom.get("State Agency Performing Data Updates"):
        key_value_tags["agency"] = sanitize_tag_key(agency_custom.get("State Agency Performing Data Updates"))[:255]

    for k, v in key_value_tags.items():
        clean_k = sanitize_tag_key(k)
        clean_v = sql_str(v)
        if clean_k and clean_v:
            tag_assignments.append(f"'{sql_str(clean_k)}' = '{clean_v}'")

    remaining_slots = MAX_TAGS - len(tag_assignments)
    for tag in unique_simple_tags[:remaining_slots]:
        tag_assignments.append(f"'{sql_str(tag)}' = ''")

    # Step A: Wipe old tags
    try:
        existing_tags_df = spark.sql(f"""
            SELECT tag_name 
            FROM `{CATALOG_NAME.lower()}`.information_schema.table_tags
            WHERE lower(catalog_name) = '{CATALOG_NAME.lower()}'
            AND lower(schema_name) = '{SCHEMA_NAME.lower()}'
            AND lower(table_name) = '{table_name.lower()}'
        """)
        existing_tag_keys = [row["tag_name"] for row in existing_tags_df.collect()]
        if existing_tag_keys:
            unset_clause = ", ".join(f"'{sql_str(k)}'" for k in existing_tag_keys)
            spark.sql(f"ALTER TABLE {TARGET_TABLE_PATH} UNSET TAGS ({unset_clause})")
            log(f"Wiped {len(existing_tag_keys)} existing tags.")
    except Exception as wipe_err:
        log(f"Notice during tag wipe: {wipe_err}")

    # Step B: Set new table tags
    if tag_assignments:
        set_tags_sql = f"ALTER TABLE {TARGET_TABLE_PATH} SET TAGS (\n    {', '.join(tag_assignments)}\n)"
        spark.sql(set_tags_sql)
        log(f"Applied {len(tag_assignments)} table tags.")

    # 7. Apply Column Comments
    for col in dataset_metadata.get("columns", []):
        col_name = col.get("fieldName")
        cmt = col.get("description") or col.get("name") or col.get("fieldName")
        try:
            spark.sql(f"""
                ALTER TABLE {TARGET_TABLE_PATH} 
                ALTER COLUMN `{col_name}` COMMENT '{sql_str(cmt)}'
            """)
        except Exception as col_err:
            log(f"Skipping comment on column `{col_name}`: {col_err}")

    # 8. Record Success
    append_log_event(
        dataset_id=DATASET_ID,
        title=dataset_title,
        table_name=table_name,
        category=category,
        portal_meta_update=PORTAL_METADATA_UPDATED_AT,
        status="SUCCESS",
        records=1,
        error=None
    )
    log(f"Completed table properties, table tags, and column comment sync for {TARGET_TABLE_PATH}.")

except Exception as err:
    log(f"CRITICAL: Task failed for dataset '{DATASET_ID}'. Logging failure to queue.")
    
    # Logs the failure with guaranteed fallback values if metadata was never resolved
    append_log_event(
        dataset_id=DATASET_ID,
        title=dataset_title,
        table_name=table_name,
        category=category,
        portal_meta_update=PORTAL_METADATA_UPDATED_AT,
        status="FAILED",
        records=0,
        error=f"{type(err).__name__}: {str(err)}"
    )
    # Fail the notebook run after writing the audit row
    raise err

finally:
    if client:
        try:
            client.close()
        except Exception:
            pass